#### **Notebook #2**
The notebook describes steps required for denoising sigma0 observation from Sentinel-1 using ESA or NERSC algorithm 

In [1]:
from s1denoise.tools import run_correction
from nansat import Nansat
import matplotlib.pyplot as plt
import xarray as xr
import pythesint as pti
from pathlib import Path

In [2]:
# Define path/to/product.SAFE(zip)
src = '/src/.devcontainer/S1A_IW_GRDH_1SDV_20220110T173429_20220110T173454_041401_04EC3B_31E4.SAFE'

In [3]:
# Apply denoisin
s1_corr = run_correction(src)

Correct VH band


/opt/conda/lib/python3.11/site-packages/s1denoise/tools.py:55: RuntimeWarning: divide by zero encountered in log10
  array = 10 * np.log10(array) - scale[pol] * (inc - angular_offset)


Correct VV band


In [4]:
s1_corr.resize(factor=0.1)
lon_grd, lats_grd = s1_corr.get_geolocation_grids()

In [7]:
n = Nansat(src)
n.resize(factor=0.1)


0.1

In [9]:
# Set up metadata and save to separate object
incidence_meta = pti.get_cf_standard_name('angle_of_incidence')
incidence_meta['_FillValue'] = -999.

lon_meta = pti.get_cf_standard_name('longitude')
lon_meta['_FillValue'] = -999.

lat_meta = pti.get_cf_standard_name('latitude')
lat_meta['_FillValue'] = -999.
             
ds = xr.Dataset(attrs=s1_corr.get_metadata(), 
                coords=dict(
                    lat=(('row', 'col'), lats_grd, lat_meta),
                    lon=(('row', 'col'), lon_grd, lon_meta)),                  
                data_vars=dict( 
                    sigma0=(('row', 'col'), s1_corr['sigma0_VV'], {'polarization': 'VV', '_FillValue': -999.}),
                    angle_of_incidence=(('row', 'col'), n['incidence_angle'], incidence_meta)
                ))

dst_path = Path(n.name).with_suffix('.nc')
if dst_path.exists(): dst_path.unlink()
ds.to_netcdf(Path(n.name).with_suffix('.nc'))
ds.close()